# Experiment Evaluation Tables

## Goal

Find every `pt_experiment.csv` below the project's `outputs` directory and display one table per experiment. Every result row is retained, while only run identifiers, selected retrieval metrics, and their related statistical columns are shown.

## Setup

Edit `SELECTED_METRICS` to change the displayed measures. When `INCLUDE_RELATED_COLUMNS` is `True`, columns such as `ndcg_cut_10 p-value`, `ndcg_cut_10 reject`, `ndcg_cut_10 +`, and `ndcg_cut_10 -` are included automatically when present.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# Project evaluation metrics. Add or remove names here as needed.
SELECTED_METRICS = [
    "ndcg_cut_10",
    "AP(rel=2)",
    "RR(rel=2)",
    # "RR@10",
    # "recall_100",
]
IDENTIFIER_COLUMNS = ["name", "run", "system", "model", "model_id", "dimension"]
INCLUDE_RELATED_COLUMNS = False
EVALUATION_FILENAME = "pt_experiment.csv"

## Discover evaluation files

The root lookup works whether the notebook is started from the repository root or from `analysis_viz`.

In [2]:
def find_project_root(start: Path) -> Path:
    """Find the nearest parent containing the codebase directory."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "matryoshka_optimization_codebase").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from inside the Master-Thesis repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUTS_DIR = PROJECT_ROOT / "matryoshka_optimization_codebase" / "outputs"
evaluation_files = sorted(OUTPUTS_DIR.rglob(EVALUATION_FILENAME)) if OUTPUTS_DIR.exists() else []

print(f"Outputs directory: {OUTPUTS_DIR}")
print(f"Found {len(evaluation_files)} evaluation file(s).")
for evaluation_file in evaluation_files:
    print(" -", evaluation_file.relative_to(OUTPUTS_DIR))

Outputs directory: /home/franchini/Master-Thesis/matryoshka_optimization_codebase/outputs
Found 17 evaluation file(s).
 - modernbert_finetuned_beir/nq/beir_nq_cosine_residual_l1_1gb/pt_experiment.csv
 - modernbert_finetuned_beir/nq/beir_nq_cosine_residual_l1_2gb/pt_experiment.csv
 - modernbert_finetuned_msmarco/devsmall_dl19_dot_logrel_bm25_dfull_8gb_oldopt/pt_experiment.csv
 - modernbert_finetuned_msmarco/devsmall_dl19_dot_logrel_bm25_tailres_8gb/pt_experiment.csv
 - modernbert_finetuned_msmarco/devsmall_dl19_dot_logrel_bm25_tailres_8gb_oldopt/pt_experiment.csv
 - modernbert_finetuned_msmarco/dl19_cosine_residual_l1_1gb/pt_experiment.csv
 - modernbert_finetuned_msmarco/dl19_cosine_residual_l1_3gb/pt_experiment.csv
 - modernbert_finetuned_msmarco/dl19_cosine_residual_l1_5gb/pt_experiment.csv
 - modernbert_finetuned_msmarco/dl19_cosine_residual_l1_8gb/pt_experiment.csv
 - modernbert_finetuned_msmarco/dl19_cosine_residual_l2_8gb/pt_experiment.csv
 - modernbert_finetuned_msmarco/dl19_dl20

## Results

Each CSV is loaded independently. The experiment label is its directory path relative to `outputs`, so nested experiments and repeated folder names remain distinguishable. A canonical column list is built across all files and applied to every table, guaranteeing identical columns in identical order; unavailable values appear as `NaN`.

In [3]:
def build_canonical_columns(column_sets: list[list[str]]) -> list[str]:
    """Build one deterministic display schema shared by every experiment table."""
    available_columns = {column for columns in column_sets for column in columns}
    canonical_columns = [column for column in IDENTIFIER_COLUMNS if column in available_columns]

    for metric in SELECTED_METRICS:
        if metric in available_columns:
            canonical_columns.append(metric)
        if INCLUDE_RELATED_COLUMNS:
            related_columns = sorted(
                column for column in available_columns if column.startswith(f"{metric} ")
            )
            canonical_columns.extend(related_columns)

    return list(dict.fromkeys(canonical_columns))


# First load every file so one common column schema can be computed.
loaded_results: dict[str, pd.DataFrame] = {}
load_errors: dict[str, str] = {}
for evaluation_file in evaluation_files:
    experiment = evaluation_file.parent.relative_to(OUTPUTS_DIR).as_posix()
    try:
        loaded_results[experiment] = pd.read_csv(evaluation_file)
    except Exception as exc:
        load_errors[experiment] = str(exc)

canonical_columns = build_canonical_columns(
    [results.columns.tolist() for results in loaded_results.values()]
)
results_by_experiment: dict[str, pd.DataFrame] = {}
discovery_rows = []

if not evaluation_files:
    display(Markdown(f"> No `{EVALUATION_FILENAME}` files were found below `{OUTPUTS_DIR}`."))
else:
    print("Canonical column order:", canonical_columns)
    for evaluation_file in evaluation_files:
        experiment = evaluation_file.parent.relative_to(OUTPUTS_DIR).as_posix()
        display(Markdown(f"### `{experiment}`"))

        if experiment in load_errors:
            error = load_errors[experiment]
            discovery_rows.append(
                {
                    "experiment": experiment,
                    "rows": None,
                    "displayed_columns": ", ".join(canonical_columns) or "(none)",
                    "missing_metrics": "unknown",
                    "status": f"read error: {error}",
                }
            )
            display(Markdown(f"> Could not read `{evaluation_file.name}`: `{error}`"))
            continue

        full_results = loaded_results[experiment]
        # reindex enforces the same columns and order; absent columns become NaN.
        selected_results = full_results.reindex(columns=canonical_columns).copy()
        results_by_experiment[experiment] = selected_results
        missing_metrics = [metric for metric in SELECTED_METRICS if metric not in full_results.columns]
        discovery_rows.append(
            {
                "experiment": experiment,
                "rows": len(full_results),
                "displayed_columns": ", ".join(canonical_columns) or "(none)",
                "missing_metrics": ", ".join(missing_metrics) or "(none)",
                "status": "ok" if canonical_columns else "no selected columns found",
            }
        )

        if canonical_columns:
            display(selected_results)
        else:
            display(Markdown("> None of the configured identifier or metric columns exists in any file."))
            print("Available columns:", full_results.columns.tolist())

Canonical column order: ['name', 'ndcg_cut_10', 'AP(rel=2)', 'RR(rel=2)']


### `modernbert_finetuned_beir/nq/beir_nq_cosine_residual_l1_1gb`

,name,ndcg_cut_10,AP(rel=2),RR(rel=2)
0,full_run,0.3411,0.0,0.0
1,profile_d512_run,0.3360,0.0,0.0
2,profile_d256_run,0.3254,0.0,0.0
3,profile_d128_run,0.3132,0.0,0.0
4,profile_d64_run,0.2918,0.0,0.0
5,profile_d32_run,0.2278,0.0,0.0
6,optimized_run,0.3032,0.0,0.0


### `modernbert_finetuned_beir/nq/beir_nq_cosine_residual_l1_2gb`

,name,ndcg_cut_10,AP(rel=2),RR(rel=2)
0,full_run,0.3411,0.0,0.0
1,profile_d512_run,0.3360,0.0,0.0
2,profile_d256_run,0.3254,0.0,0.0
3,profile_d128_run,0.3132,0.0,0.0
4,profile_d64_run,0.2918,0.0,0.0
5,profile_d32_run,0.2278,0.0,0.0
6,optimized_run,0.3163,0.0,0.0


...

,name,ndcg_cut_10,AP(rel=2),RR(rel=2)
0,full_run,0.4864,0.2498,0.7327
1,profile_d512_run,0.4807,0.2407,0.7206
2,profile_d256_run,0.4772,0.2399,0.7283
3,profile_d128_run,0.4680,0.2340,0.7234
4,profile_d64_run,0.4581,0.2249,0.7216
5,profile_d32_run,0.3779,0.1684,0.6369
6,optimized_run,0.4509,0.2180,0.6946


### `modernbert_finetuned_msmarco/dl20_cosine_residual_l1_8gb`

,name,ndcg_cut_10,AP(rel=2),RR(rel=2)
0,full_run,0.4864,0.2498,0.7327
1,profile_d512_run,0.4807,0.2407,0.7206
2,profile_d256_run,0.4772,0.2399,0.7283
3,profile_d128_run,0.4680,0.2340,0.7234
4,profile_d64_run,0.4581,0.2249,0.7216
5,profile_d32_run,0.3779,0.1684,0.6369
6,optimized_run,0.4736,0.2374,0.7225


### `modernbert_finetuned_msmarco/dl20_dot_residual_l1_8gb`

,name,ndcg_cut_10,AP(rel=2),RR(rel=2)
0,full_run,0.4864,0.2498,0.7327
1,profile_d512_run,0.4836,0.2452,0.7238
2,profile_d256_run,0.4774,0.2409,0.7277
3,profile_d128_run,0.4491,0.2295,0.6966
4,profile_d64_run,0.4155,0.2009,0.6646
5,profile_d32_run,0.2382,0.0981,0.4205
6,optimized_run,0.2975,0.1153,0.5146


## Checks

This compact summary confirms the number of rows retained from each experiment and reports missing configured metrics or unreadable files.

In [4]:
discovery_summary = pd.DataFrame(
    discovery_rows,
    columns=["experiment", "rows", "displayed_columns", "missing_metrics", "status"],
)

if discovery_summary.empty:
    print("Nothing to validate until evaluation files are available.")
else:
    display(discovery_summary)

,experiment,rows,displayed_columns,missing_metrics,status
0,modernbert_finetuned_beir/nq/beir_nq_cosine_re...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
1,modernbert_finetuned_beir/nq/beir_nq_cosine_re...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
2,modernbert_finetuned_msmarco/devsmall_dl19_dot...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
3,modernbert_finetuned_msmarco/devsmall_dl19_dot...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
4,modernbert_finetuned_msmarco/devsmall_dl19_dot...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
5,modernbert_finetuned_msmarco/dl19_cosine_resid...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
6,modernbert_finetuned_msmarco/dl19_cosine_resid...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
7,modernbert_finetuned_msmarco/dl19_cosine_resid...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
8,modernbert_finetuned_msmarco/dl19_cosine_resid...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
9,modernbert_finetuned_msmarco/dl19_cosine_resid...,7,"name, ndcg_cut_10, AP(rel=2), RR(rel=2)",(none),ok
